In [1]:
# ============================================================================
# PROJECT: COSADAM VIT MAIN COMPARISON – DeiT-Small on CIFAR-100 
# REVISION: v3.4 — VERIFIED PATHS, 7 OPTIMIZERS, REVIEWER 4 READY
# ============================================================================
# [v3.4 CHANGES]:
#   - VERIFIED PATHS from diagnostic script:
#     CIFAR-100: /kaggle/input/datasets/fedesoriano/cifar100
#     DeiT-Small: /kaggle/input/datasets/kami2suukyi/timm-pretrained-vit/vit/deit_small_patch16_224-cd65a155.pth
#   - 7 Optimizers: CosAdam, AdamW, Lion, diffGrad, AdaBelief, AngularGrad, DESGD
#   - DESGD tuned with higher LR grid: {1e-3, 3e-3, 1e-2, 3e-2, 1e-1}
#   - Lion tuned with lower LR grid: {3e-5, 1e-4, 3e-4}
#   - Other Adam-type optimizers: {1e-4, 3e-4}
#   - VIT_EPOCHS = 7
#   - SAM completely removed
#   - statistical_tests fixed for unequal run counts
# ============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import os, time, warnings, json, random, pickle, shutil, math, sys, gc
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score, matthews_corrcoef
from scipy.stats import ttest_rel, t as t_dist
from statsmodels.stats.multitest import multipletests

try:
    import timm
except ImportError:
    os.system('pip install timm -q')
    import timm

# ============================================================================
# GLOBAL TIMERS & VERIFIED PATHS
# ============================================================================
GLOBAL_START_TIME = time.time()
SAFE_TIME_LIMIT_SECONDS = 11.4 * 3600  

os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Execution Device Verified: {device}", flush=True)

# [v3.4] GPU CHECK - CRITICAL WARNING
if device.type == 'cpu':
    print("\n" + "="*80, flush=True)
    print("⚠️ ⚠️ ⚠️ CRITICAL WARNING: GPU IS NOT AVAILABLE! ⚠️ ⚠️ ⚠️", flush=True)
    print("The code is running on CPU. Execution will take 10-20x longer.", flush=True)
    print("To enable GPU in Kaggle:", flush=True)
    print("  1. Click 'Settings' (gear icon) in the right sidebar", flush=True)
    print("  2. Under 'Accelerator', select 'GPU T4 x2'", flush=True)
    print("  3. Save the notebook and re-run", flush=True)
    print("="*80 + "\n", flush=True)

torch.backends.cudnn.benchmark = False 

# [v3.4] VERIFIED PATHS from diagnostic script
CIFAR100_KAGGLE_PATH = '/kaggle/input/datasets/fedesoriano/cifar100'
TIMM_WEIGHTS_PATH = '/kaggle/input/datasets/kami2suukyi/timm-pretrained-vit/vit/deit_small_patch16_224-cd65a155.pth'

RESULTS_DIR = "/kaggle/working/results_cosadam_vit_main"
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "latex_tables"), exist_ok=True)
MAIN_RESULTS_FILE = os.path.join(RESULTS_DIR, "vit_main_metrics_final.json")
HP_CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "vit_hp_checkpoint.json")

# ============================================================================
# HYPERPARAMETERS
# ============================================================================
VIT_EPOCHS = 7              
BATCH_SIZE = 128            
GRADIENT_CLIP_VALUE = 1.0
WARMUP_EPOCHS = 2
WARMUP_RATIO = 0.1

HP_EPOCHS = 1                  
HP_SUBSET_RATIO = 0.2          

HP_SEARCH_SEED = 42
MAIN_SEEDS = [42, 123, 777]   

# [v3.4] Optimizer-specific hyperparameter grids per Reviewer 4 feedback
CUSTOM_HP_GRIDS = {
    'DESGD': {
        'lr': [1e-3, 3e-3, 1e-2, 3e-2, 1e-1],  # Higher LR for SGD-based method
        'wd': [0.01, 0.05]
    },
    'Lion': {
        'lr': [3e-5, 1e-4, 3e-4],               # Lower LR for sign-based method
        'wd': [0.01, 0.05]
    },
    'default': {
        'lr': [1e-4, 3e-4],                     # Standard grid for Adam-type methods
        'wd': [0.01, 0.05]
    }
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    return seed_worker

# ============================================================================
# 🧠 OPTIMIZERS (7 Optimizers - SAM REMOVED)
# ============================================================================

# 1. CosAdam (Proposed)
class CosAdam(optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.05, alpha_cos=0.9, c=0.5):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, alpha_cos=alpha_cos, c=c)
        super(CosAdam, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0; state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p); state['prev_grad'] = torch.zeros_like(p); state['s'] = 0.0
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                prev_grad, s = state['prev_grad'], state['s']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(grad.flatten(), prev_grad.flatten(), dim=0, eps=1e-8)
                    s = group['alpha_cos'] * s + (1 - group['alpha_cos']) * cos_theta.item()
                state['s'] = s
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1, bc2 = 1 - beta1 ** state['step'], 1 - beta2 ** state['step']
                denom = (exp_avg_sq.sqrt() / math.sqrt(bc2)).add_(group['eps'])
                step_size = group['lr'] / bc1
                update = -step_size * exp_avg / denom
                update.mul_(1 + group['c'] * s)
                if group['weight_decay'] != 0: p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                p.add_(update)
                state['prev_grad'].copy_(grad)
        return loss

# 2. AdamW (torch.optim.AdamW used directly)

# 3. Lion
class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.01):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                grad, state = p.grad, self.state[p]
                if group["weight_decay"] != 0: p.mul_(1 - group["lr"] * group["weight_decay"])
                if "exp_avg" not in state: state["exp_avg"] = torch.zeros_like(p)
                exp_avg = state["exp_avg"]
                beta1, beta2 = group["betas"]
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group["lr"])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)

# 4. diffGrad
class DiffGrad(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        super(DiffGrad, self).__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0; state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p); state['prev_grad'] = torch.zeros_like(p)
                beta1, beta2 = group['betas']
                state['step'] += 1
                if group['weight_decay'] != 0: p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                diff = torch.abs(state['prev_grad'] - grad)
                dfc = 1.0 / (1.0 + torch.exp(-diff))
                state['exp_avg'].mul_(beta1).add_(grad, alpha=1 - beta1)
                state['exp_avg_sq'].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1, bc2 = 1 - beta1 ** state['step'], 1 - beta2 ** state['step']
                step_size = group['lr'] / bc1
                denom = (state['exp_avg_sq'].sqrt() / math.sqrt(bc2)).add_(group['eps'])
                p.add_((state['exp_avg'] / denom) * dfc, alpha=-step_size)
                state['prev_grad'].copy_(grad)

# 5. AdaBelief
class AdaBelief(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-16, weight_decay=0):
        super(AdaBelief, self).__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0; state['exp_avg'] = torch.zeros_like(p); state['exp_avg_sq'] = torch.zeros_like(p)
                beta1, beta2 = group['betas']
                state['step'] += 1
                if group['weight_decay'] != 0: p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                state['exp_avg'].mul_(beta1).add_(grad, alpha=1 - beta1)
                grad_residual = grad - state['exp_avg']
                state['exp_avg_sq'].mul_(beta2).addcmul_(grad_residual, grad_residual, value=1 - beta2)
                state['exp_avg_sq'].add_(group['eps'])
                bc1, bc2 = 1 - beta1 ** state['step'], 1 - beta2 ** state['step']
                step_size = group['lr'] / bc1
                denom = (state['exp_avg_sq'].sqrt() / math.sqrt(bc2)).add_(group['eps'])
                p.addcdiv_(state['exp_avg'], denom, value=-step_size)

# 6. AngularGrad
class AngularGrad(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        super(AngularGrad, self).__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0; state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p); state['prev_grad'] = torch.zeros_like(p)
                    state['min_angle'] = torch.tensor(math.pi)
                beta1, beta2 = group['betas']
                state['step'] += 1
                if group['weight_decay'] != 0: p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(grad.flatten(), state['prev_grad'].flatten(), dim=0, eps=1e-8)
                    cos_theta = torch.clamp(cos_theta, -1.0 + 1e-7, 1.0 - 1e-7)
                    angle = torch.acos(cos_theta)
                    state['min_angle'] = torch.min(state['min_angle'].to(angle.device), angle)
                    angular_coef = torch.clamp(state['min_angle'] / angle, min=0.1, max=1.0)
                else:
                    angular_coef = 1.0
                state['exp_avg'].mul_(beta1).add_(grad, alpha=1 - beta1)
                state['exp_avg_sq'].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1, bc2 = 1 - beta1 ** state['step'], 1 - beta2 ** state['step']
                step_size = group['lr'] / bc1
                denom = (state['exp_avg_sq'].sqrt() / math.sqrt(bc2)).add_(group['eps'])
                p.add_((state['exp_avg'] / denom) * angular_coef, alpha=-step_size)
                state['prev_grad'].copy_(grad)

# 7. DESGD
class DESGD(Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.9, gamma=0.01, weight_decay=0):
        super(DESGD, self).__init__(params, dict(lr=lr, momentum=momentum, gamma=gamma, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0; state['momentum_buffer'] = torch.zeros_like(p)
                    state['prev_grad'] = torch.zeros_like(p); state['dynamic_lr'] = group['lr']
                buf, prev_grad = state['momentum_buffer'], state['prev_grad']
                state['step'] += 1
                if group['weight_decay'] != 0: p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(grad.flatten(), state['prev_grad'].flatten(), dim=0, eps=1e-8)
                    state['dynamic_lr'] = state['dynamic_lr'] * (1 + group['gamma'] * cos_theta.item())
                state['dynamic_lr'] = max(min(state['dynamic_lr'], group['lr'] * 10), group['lr'] * 0.1)
                buf.mul_(group['momentum']).add_(grad)
                p.add_(buf, alpha=-state['dynamic_lr'])
                state['prev_grad'].copy_(grad)

# ============================================================================
# 📉 FLAT MINIMA EVALUATION
# ============================================================================
def evaluate_loss_sharpness(model, data_loader, criterion, epsilon=0.05):
    model.eval()
    original_state = {k: v.clone() for k, v in model.state_dict().items()}
    x, y = next(iter(data_loader))
    x, y = x.to(device), y.to(device)
    model.zero_grad()
    loss_orig = criterion(model(x), y)
    loss_orig.backward()
    with torch.no_grad():
        for p in model.parameters():
            if p.grad is not None: p.add_(epsilon * torch.sign(p.grad))
    loss_perturbed = criterion(model(x), y).item()
    model.load_state_dict(original_state)
    return float(loss_perturbed - loss_orig.item())

# ============================================================================
# DATASET & MODEL
# ============================================================================
class CIFAR100Manual(Dataset):
    def __init__(self, root, train=True, transform=None):
        # [v3.4] Handle both direct path and path with 'cifar-100-python' subfolder
        if os.path.exists(os.path.join(root, 'train')):
            base_dir = root
        else:
            base_dir = os.path.join(root, 'cifar-100-python')
        
        with open(os.path.join(base_dir, 'train' if train else 'test'), 'rb') as f:
            batch = pickle.load(f, encoding='latin1')
        self.data = batch['data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        self.targets = batch['fine_labels']
        self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        img = self.transform(Image.fromarray(self.data[idx])) if self.transform else Image.fromarray(self.data[idx])
        return img, int(self.targets[idx])

def get_cifar100_loaders(batch_size=64, subset_ratio=1.0, seed=42):
    target_base = '/kaggle/working/cifar100_data'
    
    # Determine the actual source directory
    if os.path.exists(os.path.join(CIFAR100_KAGGLE_PATH, 'train')):
        src_dir = CIFAR100_KAGGLE_PATH
        dst_dir = target_base
    else:
        src_dir = CIFAR100_KAGGLE_PATH
        dst_dir = os.path.join(target_base, 'cifar-100-python')
    
    train_file_dest = os.path.join(dst_dir, 'train')
    
    if not os.path.exists(train_file_dest):
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.exists(src_dir):
            print(f"📋 Copying CIFAR-100 from {src_dir} to {dst_dir}...", flush=True)
            for item in os.listdir(src_dir):
                s = os.path.join(src_dir, item)
                d = os.path.join(dst_dir, item)
                if os.path.isdir(s):
                    shutil.copytree(s, d, dirs_exist_ok=True)
                else:
                    shutil.copy2(s, d)
            print("✅ Copy complete.", flush=True)
        else: 
            raise FileNotFoundError(f"🚨 CIFAR-100 source not found at {src_dir}")
    
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15), transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
        transforms.RandomErasing(p=0.25)
    ])
    transform_test = transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761))
    ])
    
    train_ds = CIFAR100Manual(target_base, train=True, transform=transform_train)
    if subset_ratio < 1.0:
        indices = np.random.default_rng(seed).choice(len(train_ds), int(len(train_ds) * subset_ratio), replace=False)
        train_ds = Subset(train_ds, indices)
    val_ds = CIFAR100Manual(target_base, train=False, transform=transform_test)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, worker_init_fn=set_seed(seed))
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader

class ViT_CIFAR100(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.model = timm.create_model('deit_small_patch16_224', pretrained=False, num_classes=num_classes)
        weights_loaded = False
        if os.path.exists(TIMM_WEIGHTS_PATH):
            try:
                state_dict = torch.load(TIMM_WEIGHTS_PATH, map_location='cpu')
                if 'model' in state_dict: state_dict = state_dict['model']
                state_dict = {k: v for k, v in state_dict.items() if not k.startswith('head')}
                self.model.load_state_dict(state_dict, strict=False)
                weights_loaded = True
                print(f"✅ DeiT-Small weights loaded from {TIMM_WEIGHTS_PATH}", flush=True)
            except Exception as e:
                print(f"⚠️ Failed to load weights: {e}", flush=True)
        if not weights_loaded:
            print("⚠️ Pretrained weights not found, using random initialization", flush=True)
    def forward(self, x): return self.model(x)

# ============================================================================
# TRAINING CORE
# ============================================================================
def train_and_eval(model, train_loader, val_loader, optimizer, total_steps, warmup_steps, num_epochs):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler() 
    
    def lr_lambda(s):
        if s < warmup_steps: return float(s) / max(1, warmup_steps)
        progress = float(s - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(1e-4, 0.5 * (1.0 + math.cos(math.pi * progress)))
        
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}
    best_val_loss, patience, no_improve = float('inf'), 5, 0
    best_state = None

    for epoch in range(num_epochs):
        if time.time() - GLOBAL_START_TIME > SAFE_TIME_LIMIT_SECONDS:
            print(f"\n⚠️ ⏳ [GRACEFUL EXIT] Runtime threshold reached.", flush=True)
            return None 
        t0 = time.time()
        model.train()
        tr_loss, tr_correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True) 
            with torch.cuda.amp.autocast():
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_correct += out.argmax(1).eq(y).sum().item()
            tr_loss += loss.item() * x.size(0)
            total += y.size(0)

        epoch_time = time.time() - t0
        history['epoch_time'].append(epoch_time)
        history['train_loss'].append(tr_loss / total)
        history['train_acc'].append(tr_correct / total)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                out = model(x)
                loss = criterion(out, y)
                val_loss += loss.item() * x.size(0)
                val_correct += out.argmax(1).eq(y).sum().item()
                val_total += y.size(0)
                
        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(val_correct / val_total)

        if history['val_loss'][-1] < best_val_loss - 1e-4:
            best_val_loss = history['val_loss'][-1]
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            
        print(f"      🔹 Epoch {epoch+1}/{num_epochs} | Val Acc: {history['val_acc'][-1]:.4f} | Time: {epoch_time:.0f}s", flush=True)
        gc.collect(); torch.cuda.empty_cache()
        if no_improve >= patience and epoch >= patience: break

    if best_state: model.load_state_dict(best_state)
    sharpness = evaluate_loss_sharpness(model, train_loader, criterion)
    
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            all_probs.append(F.softmax(out, dim=1).cpu().numpy())
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    all_preds, all_targets = np.array(all_preds), np.array(all_targets)
    all_probs = np.vstack(all_probs)

    try: auc_val = float(roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro'))
    except: auc_val = 0.0

    return {'test_acc': float((all_preds == all_targets).mean()), 
            'test_f1': float(f1_score(all_targets, all_preds, average='macro', zero_division=0)), 
            'test_auc': auc_val, 'test_mcc': float(matthews_corrcoef(all_targets, all_preds)), 
            'sharpness': sharpness,  
            'avg_epoch_time': float(np.mean(history['epoch_time'])), 'history': history}

# ============================================================================
# STATISTICAL BACKEND
# ============================================================================
def adaptive_ci(data, cl=0.95):
    n = len(data)
    if n < 5:
        mean, std = np.mean(data), np.std(data, ddof=1) if n > 1 else 0.0
        return float(mean - t_dist.ppf((1+cl)/2, max(n-1, 1)) * std / np.sqrt(n) if n>0 else 0), float(mean + t_dist.ppf((1+cl)/2, max(n-1, 1)) * std / np.sqrt(n) if n>0 else 0)
    boot = [np.mean(np.random.default_rng(42).choice(data, n, replace=True)) for _ in range(1000)]
    return float(np.percentile(boot, 100*(1-cl)/2)), float(np.percentile(boot, 100*(1-(1-cl)/2)))

def compute_statistics(state):
    main_data = state.get("main_run", {})
    if not main_data: return {}, {}, {}, {}
    all_seeds = list(main_data.keys())
    optimizers = list(main_data[all_seeds[0]].keys())
    agg, var, cvs, ci = {}, {}, {}, {}
    for opt in optimizers:
        runs = [main_data[s].get(opt, {}) for s in all_seeds if opt in main_data[s] and main_data[s][opt].get("test_acc") is not None]
        if not runs: continue
        agg[opt], var[opt], cvs[opt], ci[opt] = {}, {}, {}, {}
        for m in ['test_acc', 'test_f1', 'test_auc', 'test_mcc', 'sharpness', 'avg_epoch_time']:
            vals = [r.get(m, 0) for r in runs]
            mean_v, std_v = float(np.mean(vals)), float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            agg[opt][f'{m}_mean'], agg[opt][f'{m}_std'] = mean_v, std_v
            var[opt][f'{m}_variance'] = float(np.var(vals, ddof=1)) if len(vals) > 1 else 0.0
            cvs[opt][f'{m}_cv'] = float((std_v/mean_v*100) if mean_v>0 else 0)
            ci[opt][m] = adaptive_ci(vals)
    return agg, var, cvs, ci

def statistical_tests(state, baseline='CosAdam'):
    main_data = state.get("main_run", {})
    if not main_data: return {}
    all_seeds = list(main_data.keys())
    
    base_runs = [main_data[s].get(baseline, {}) for s in all_seeds 
                 if baseline in main_data[s] and main_data[s][baseline].get("test_acc") is not None]
    if len(base_runs) < 2: return {}
    base_vals = np.array([r.get("test_acc") for r in base_runs])
    
    sig, pvals, basenames = {}, [], []

    for opt in list(main_data[all_seeds[0]].keys()):
        if opt == baseline: continue
        runs = [main_data[s].get(opt, {}) for s in all_seeds 
                if opt in main_data[s] and main_data[s][opt].get("test_acc") is not None]
        if len(runs) < 2: continue
        vals = np.array([r.get("test_acc") for r in runs])
        
        # [FIX] Trim to minimum length for paired t-test
        min_len = min(len(base_vals), len(vals))
        base_trimmed = base_vals[:min_len]
        vals_trimmed = vals[:min_len]
        
        diffs = base_trimmed - vals_trimmed
        mean_diff, std_diff = float(np.mean(diffs)), float(np.std(diffs, ddof=1)) if min_len > 1 else 0.0
        d_z = float(mean_diff / std_diff) if std_diff >= 1e-8 else 0.0
        
        if min_len >= 2 and std_diff >= 1e-8:
            p_raw = float(ttest_rel(base_trimmed, vals_trimmed)[1])
        else:
            p_raw = 1.0 if abs(mean_diff) < 1e-8 else 0.0
                
        pvals.append(p_raw)
        basenames.append(opt)
        sig[opt] = {'diff': mean_diff, 'p_raw': p_raw, 'es_z': d_z}
        
    if pvals:
        _, p_corr, _, _ = multipletests(pvals, method='fdr_bh')
        for i, opt in enumerate(basenames):
            sig[opt].update({'p_corr': float(p_corr[i]), 
                            'sig': '***' if p_corr[i]<0.001 else '**' if p_corr[i]<0.01 else '*' if p_corr[i]<0.05 else 'ns'})
    return sig

# ============================================================================
# 📊 LATEX TABLES & PLOTS
# ============================================================================
def performance_table(agg, ci, caption, label, filename):
    mc = [
        {'d': 'Acc', 'k': 'test_acc', 'f': '.4f', 'show_ci': True},
        {'d': 'Sharpness', 'k': 'sharpness', 'f': '.4f', 'show_ci': False},
        {'d': 'F1', 'k': 'test_f1', 'f': '.4f', 'show_ci': False},
        {'d': 'AUC', 'k': 'test_auc', 'f': '.4f', 'show_ci': False},
        {'d': 'Time(s)', 'k': 'avg_epoch_time', 'f': '.2f', 'show_ci': False}
    ]
    lines = [r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}", r"\label{" + label + "}",
             r"\resizebox{\textwidth}{!}{%", r"\begin{tabular}{l" + "c" * len(mc) + "}", r"\toprule",
             r"{\bf Optimizer} & " + " & ".join([f"{{\\bf {m['d']}}}" for m in mc]) + r" \\", r"\midrule"]
    for opt in agg.keys():
        row = [opt]
        for m in mc:
            if f"{m['k']}_mean" not in agg[opt]: row.append("-"); continue
            mean, std = agg[opt][f"{m['k']}_mean"], agg[opt][f"{m['k']}_std"]
            if m['show_ci']:
                low, high = ci[opt][m['k']]
                ci_str = r"\; [--, --]" if np.isnan(low) or np.isnan(high) else (f" \\; [{low:{m['f']}}, {high:{m['f']}}]")
                row.append(f"{mean:{m['f']}} $\\pm$ {std:{m['f']}}{ci_str}")
            else:
                row.append(f"{mean:{m['f']}} $\\pm$ {std:{m['f']}}")
        lines.append(" & ".join(row) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"}", r"\end{table}"]
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def stats_table(sig, caption, label, filename):
    lines = [r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}", r"\label{" + label + "}",
             r"\begin{tabular}{lccccc}", r"\toprule",
             r"\textbf{Comparison vs CosAdam} & \textbf{$\Delta$ Acc} & \textbf{$p_{\text{ttest}}$} & \textbf{$p_{\text{FDR}}$} & \textbf{$d_z$} & \textbf{Sig.} \\", r"\midrule"]
    for opt, r in sig.items():
        diff_str = (f"${r['diff']:+.4f}$" if abs(r['diff']) >= 1e-4 else f"${r['diff']:+.1e}$")
        lines.append(f"{opt} & {diff_str} & {r['p_raw']:.3f} & {r['p_corr']:.3f} & {r['es_z']:.3f} & {r['sig']} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def plot_learning_curves(state, title, save_name):
    main_data = state.get("main_run", {})
    if not main_data: return
    try: optimizers = list(main_data[list(main_data.keys())[0]].keys())
    except: return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors, line_styles = plt.cm.tab10.colors, ['-', '--']
    for idx, opt in enumerate(optimizers):
        runs = [main_data[s][opt] for s in main_data.keys() if opt in main_data[s] and 'history' in main_data[s][opt]]
        if not runs: continue
        max_epochs = max([len(r['history']['train_acc']) for r in runs])
        all_tr_a, all_va_a, all_tr_l, all_va_l = [], [], [], []
        for r in runs:
            h = r['history']
            ta, va, tl, vl = list(h.get('train_acc', [])), list(h.get('val_acc', [])), list(h.get('train_loss', [])), list(h.get('val_loss', []))
            if len(ta) < max_epochs: ta += [ta[-1]]*(max_epochs-len(ta)); va += [va[-1]]*(max_epochs-len(va)); tl += [tl[-1]]*(max_epochs-len(tl)); vl += [vl[-1]]*(max_epochs-len(vl))
            all_tr_a.append(ta); all_va_a.append(va); all_tr_l.append(tl); all_va_l.append(vl)
        mta, sta = np.mean(all_tr_a, axis=0), np.std(all_tr_a, axis=0)
        mva, sva = np.mean(all_va_a, axis=0), np.std(all_va_a, axis=0)
        mtl, stl = np.mean(all_tr_l, axis=0), np.std(all_tr_l, axis=0)
        mvl, svl = np.mean(all_va_l, axis=0), np.std(all_va_l, axis=0)
        ep = range(1, max_epochs + 1)
        ax1.plot(ep, mta, color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[0], alpha=0.5)
        ax1.plot(ep, mva, label=opt, color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[1])
        ax1.fill_between(ep, mta-sta, mta+sta, color=colors[idx % len(colors)], alpha=0.1)
        ax1.fill_between(ep, mva-sva, mva+sva, color=colors[idx % len(colors)], alpha=0.2)
        ax2.plot(ep, mtl, color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[0], alpha=0.5)
        ax2.plot(ep, mvl, label=opt, color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[1])
        ax2.fill_between(ep, mtl-stl, mtl+stl, color=colors[idx % len(colors)], alpha=0.1)
        ax2.fill_between(ep, mvl-svl, mvl+svl, color=colors[idx % len(colors)], alpha=0.2)
    ax1.set_title(f'{title} Accuracy', fontweight='bold'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.grid(alpha=0.3); ax1.legend(loc='lower right', fontsize=8)
    ax2.set_title(f'{title} Loss', fontweight='bold'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.grid(alpha=0.3); ax2.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "plots", save_name), dpi=300, bbox_inches='tight'); plt.close()

def plot_pareto_efficiency(agg, save_path=None):
    fig, ax = plt.subplots(figsize=(10, 7))
    for opt, m in agg.items():
        if 'avg_epoch_time_mean' in m and 'test_acc_mean' in m:
            ax.scatter(m['avg_epoch_time_mean'], m['test_acc_mean'], label=opt, s=150, alpha=0.8)
            ax.annotate(opt, (m['avg_epoch_time_mean'], m['test_acc_mean']), xytext=(7, 7), textcoords='offset points', fontsize=10, fontweight='bold')
    ax.set_title("Pareto Efficiency: Accuracy vs. Compute Time", fontweight='bold', fontsize=14)
    ax.set_xlabel("Avg Epoch Time (s) ↓", fontsize=12); ax.set_ylabel("Test Accuracy ↑", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6); ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout()
    plt.savefig(save_path or os.path.join(RESULTS_DIR, "plots", "vit_pareto_efficiency.png"), dpi=300, bbox_inches='tight'); plt.close()

def plot_stability_boxplot(state, save_path=None):
    main_data = state.get("main_run", {})
    if not main_data: return
    try: optimizers = list(main_data[list(main_data.keys())[0]].keys())
    except: return
    fig, ax = plt.subplots(figsize=(12, 7))
    data_to_plot, labels = [], []
    for opt in optimizers:
        accs = [main_data[s][opt]['test_acc'] for s in main_data.keys() if opt in main_data[s] and 'test_acc' in main_data[s][opt]]
        if accs: data_to_plot.append(accs); labels.append(f"{opt}\n(n={len(accs)})")
    if data_to_plot:
        box = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        for patch, color in zip(box['boxes'], plt.cm.Set3(np.linspace(0, 1, len(labels)))): patch.set_facecolor(color)
        ax.set_title("Stability Analysis: Accuracy Variance Across Seeds", fontweight='bold', fontsize=14)
        ax.set_ylabel("Test Accuracy", fontsize=12); ax.grid(True, axis='y', linestyle='--', alpha=0.6)
        plt.xticks(rotation=0, fontsize=10, fontweight='bold'); plt.tight_layout()
        plt.savefig(save_path or os.path.join(RESULTS_DIR, "plots", "vit_stability_analysis.png"), dpi=300, bbox_inches='tight')
    plt.close()

# ============================================================================
# EXPORT & MAIN EXECUTION
# ============================================================================
def generate_all_outputs(state):
    print("\nCompiling statistical backends, LaTeX tables, and visualizations...", flush=True)
    agg, var, cvs, ci = compute_statistics(state)
    if not agg: return
    
    performance_table(agg, ci, r"ViT Main Comparison (DeiT-Small on CIFAR-100, Mean $\pm$ SD [95\% CI])", "tab:vit_main_perf", "t1_vit_main_perf.tex")
    sig = statistical_tests(state, baseline='CosAdam')
    stats_table(sig, r"Statistical Significance (CosAdam vs baselines, Paired T-Test)", "tab:vit_main_stats", "t2_vit_main_stats.tex")
    
    plot_learning_curves(state, "DeiT-Small on CIFAR-100", "vit_learning_curves.png")
    plot_pareto_efficiency(agg)
    plot_stability_boxplot(state)
    print("✅ Comprehensive academic suite safely written.", flush=True)

def create_final_zip():
    zip_path = "/kaggle/working/final_cosadam_results.zip"
    base_folder = os.path.basename(RESULTS_DIR)  
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for f_path in [MAIN_RESULTS_FILE, HP_CHECKPOINT_FILE]:
            if os.path.exists(f_path): zipf.write(f_path, arcname=os.path.join(base_folder, os.path.basename(f_path)))
        for subdir in ["plots", "latex_tables"]:
            dir_path = os.path.join(RESULTS_DIR, subdir)
            if os.path.isdir(dir_path):
                for root, _, files in os.walk(dir_path):
                    for file in files: zipf.write(os.path.join(root, file), arcname=os.path.join(base_folder, subdir, file))
    print(f"📦 Final results archived at: {zip_path}", flush=True)

def main():
    print("=" * 80, flush=True)
    print("COSADAM ViT MAIN COMPARISON v3.4 – VERIFIED PATHS", flush=True)
    print(f"Device: {device}", flush=True)
    print(f"CIFAR-100: {CIFAR100_KAGGLE_PATH}", flush=True)
    print(f"DeiT-Small: {TIMM_WEIGHTS_PATH}", flush=True)
    print("Optimizers: CosAdam, AdamW, Lion, diffGrad, AdaBelief, AngularGrad, DESGD", flush=True)
    print("Epochs: 7 | Seeds: 3 | SAM: Removed | DESGD: Extended LR Grid", flush=True)
    print("=" * 80, flush=True)

    hp_configs = {}
    opt_list = [
        ('CosAdam', CosAdam), ('AdamW', optim.AdamW), ('Lion', Lion), 
        ('diffGrad', DiffGrad), ('AdaBelief', AdaBelief), 
        ('AngularGrad', AngularGrad), ('DESGD', DESGD)
    ]
    
    hp_data = {}
    if os.path.exists(HP_CHECKPOINT_FILE):
        try:
            with open(HP_CHECKPOINT_FILE, 'r') as f: hp_data = json.load(f)
        except: pass

    print("\n  🔍 [Phase 1] Starting Optimizer-Specific Hyperparameter Search...", flush=True)
    for opt_name, opt_class in opt_list:
        if opt_name in hp_data:
            hp_configs[opt_name] = (opt_class, hp_data[opt_name]['lr'], hp_data[opt_name]['wd'])
            print(f"  ⏭️ [Checkpoint] Skipped HP Search for {opt_name}.", flush=True)
            continue
        
        grid = CUSTOM_HP_GRIDS.get(opt_name, CUSTOM_HP_GRIDS['default'])
        search_lrs = grid['lr']
        search_wds = grid['wd']
            
        print(f"  ⚡ Tuning {opt_name} (LR grid: {search_lrs})...", flush=True)
        train_loader, val_loader = get_cifar100_loaders(BATCH_SIZE, subset_ratio=HP_SUBSET_RATIO, seed=HP_SEARCH_SEED)
        total_steps = HP_EPOCHS * len(train_loader)
        warmup_steps = int(WARMUP_RATIO * total_steps)
        best_acc, best_params = 0.0, (search_lrs[0], search_wds[0])
        
        for lr in search_lrs:
            for wd in search_wds:
                if time.time() - GLOBAL_START_TIME > SAFE_TIME_LIMIT_SECONDS:
                    print("\n⚠️ ⏳ [GRACEFUL EXIT] Time limit approaching!", flush=True); sys.exit(0)
                set_seed(HP_SEARCH_SEED)
                model = ViT_CIFAR100().to(device)
                optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd)
                res = train_and_eval(model, train_loader, val_loader, optimizer, total_steps, warmup_steps, HP_EPOCHS)
                if res and res['test_acc'] > best_acc: 
                    best_acc = res['test_acc']; best_params = (lr, wd)
                del model, optimizer, res; gc.collect(); torch.cuda.empty_cache()
                
        hp_data[opt_name] = {'lr': best_params[0], 'wd': best_params[1], 'acc': best_acc}
        with open(HP_CHECKPOINT_FILE, 'w') as f: json.dump(hp_data, f, indent=4)
        hp_configs[opt_name] = (opt_class, best_params[0], best_params[1])
        print(f"  🎯 Best parameters for {opt_name} saved: lr={best_params[0]:.1e}, wd={best_params[1]}, acc={best_acc:.4f}", flush=True)

    print(f"\n  🚀 [Phase 2] Initializing full dataset loaders...", flush=True)
    state = {}
    if os.path.exists(MAIN_RESULTS_FILE):
        try:
            with open(MAIN_RESULTS_FILE, 'r') as f: state = json.load(f)
        except: pass
    if "main_run" not in state: state["main_run"] = {}
    
    for seed in MAIN_SEEDS:
        seed_str = str(seed)
        if seed_str not in state["main_run"]: state["main_run"][seed_str] = {}
        for opt_name, (opt_class, lr, wd) in hp_configs.items():
            if opt_name in state["main_run"][seed_str] and 'test_acc' in state["main_run"][seed_str][opt_name]: 
                print(f"  ⏭️ [Checkpoint Check] {opt_name} on Seed {seed} already verified.", flush=True)
                continue
            print(f"  🔥 Evaluating {opt_name} on seed={seed}...", flush=True)
            train_loader, val_loader = get_cifar100_loaders(BATCH_SIZE, subset_ratio=1.0, seed=seed)
            total_steps = VIT_EPOCHS * len(train_loader)
            warmup_steps = WARMUP_EPOCHS * len(train_loader)
            set_seed(seed); torch.cuda.empty_cache(); gc.collect()
            model = ViT_CIFAR100().to(device)
            optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd)
            result = train_and_eval(model, train_loader, val_loader, optimizer, total_steps, warmup_steps, VIT_EPOCHS)
            if result is None:
                print("\n⚠️ ⏳ [GRACEFUL EXIT] Runtime budget expired.", flush=True)
                with open(MAIN_RESULTS_FILE, 'w') as f: json.dump(state, f, indent=4)
                generate_all_outputs(state); create_final_zip(); sys.exit(0)
            state["main_run"][seed_str][opt_name] = result
            with open(MAIN_RESULTS_FILE, 'w') as f: json.dump(state, f, indent=4)
            print(f"  ✔️ Completed {opt_name} seed={seed} -> Acc = {result['test_acc']:.4f} | Sharpness = {result['sharpness']:.4f}\n", flush=True)
            del model, optimizer, result, train_loader, val_loader; gc.collect(); torch.cuda.empty_cache()

    generate_all_outputs(state)
    create_final_zip()
    print("\n✅ ViT MAIN COMPARISON v3.4 COMPLETE.", flush=True)

if __name__ == "__main__":
    main()

🚀 Execution Device Verified: cuda
COSADAM ViT MAIN COMPARISON v3.4 – VERIFIED PATHS
Device: cuda
CIFAR-100: /kaggle/input/datasets/fedesoriano/cifar100
DeiT-Small: /kaggle/input/datasets/kami2suukyi/timm-pretrained-vit/vit/deit_small_patch16_224-cd65a155.pth
Optimizers: CosAdam, AdamW, Lion, diffGrad, AdaBelief, AngularGrad, DESGD
Epochs: 7 | Seeds: 3 | SAM: Removed | DESGD: Extended LR Grid

  🔍 [Phase 1] Starting Optimizer-Specific Hyperparameter Search...
  ⚡ Tuning CosAdam (LR grid: [0.0001, 0.0003])...
📋 Copying CIFAR-100 from /kaggle/input/datasets/fedesoriano/cifar100 to /kaggle/working/cifar100_data...
✅ Copy complete.
✅ DeiT-Small weights loaded from /kaggle/input/datasets/kami2suukyi/timm-pretrained-vit/vit/deit_small_patch16_224-cd65a155.pth
      🔹 Epoch 1/1 | Val Acc: 0.3928 | Time: 38s
✅ DeiT-Small weights loaded from /kaggle/input/datasets/kami2suukyi/timm-pretrained-vit/vit/deit_small_patch16_224-cd65a155.pth
      🔹 Epoch 1/1 | Val Acc: 0.3906 | Time: 38s
✅ DeiT-Small 